In [14]:
# Block 10: Load data and prepare
import pymc as pm
import pandas as pd
import numpy as np
import arviz as az

race_data = pd.read_csv('f1_race_data_cleaned.csv')

# Get dimensions
n_drivers = race_data['DriverIdx'].nunique()
n_constructors = race_data['ConstructorIdx'].nunique()
n_constructor_seasons = race_data['ConstructorSeasonIdx'].nunique()

print(f"Drivers: {n_drivers}, Constructors: {n_constructors}, Constructor-Seasons: {n_constructor_seasons}")

Drivers: 21, Constructors: 10, Constructor-Seasons: 10


In [15]:
# Block 11.5: 清理数据中的问题
# 在 Block 11 之前运行

# 检查并填充 NaN
print("Checking for NaN values:")
print(race_data[['EffectivePosition', 'QualifyingPosition', 'Recent3Avg', 'CumDNFRate']].isnull().sum())

# 填充缺失值
race_data['Recent3Avg'] = race_data['Recent3Avg'].fillna(10.5)
race_data['CumDNFRate'] = race_data['CumDNFRate'].fillna(0.0)

# 确保没有异常值
print("\nValue ranges:")
print(f"Position: {race_data['EffectivePosition'].min()}-{race_data['EffectivePosition'].max()}")
print(f"Qualifying: {race_data['QualifyingPosition'].min()}-{race_data['QualifyingPosition'].max()}")
print(f"Recent3Avg: {race_data['Recent3Avg'].min():.1f}-{race_data['Recent3Avg'].max():.1f}")

# 删除任何有问题的行
race_data_clean = race_data[
    (race_data['EffectivePosition'].notna()) &
    (race_data['QualifyingPosition'].notna()) &
    (race_data['DriverIdx'].notna()) &
    (race_data['ConstructorIdx'].notna())
].copy()

print(f"\nRows: {len(race_data)} -> {len(race_data_clean)}")

# 用清理后的数据
race_data = race_data_clean

Checking for NaN values:
EffectivePosition      0
QualifyingPosition     1
Recent3Avg            21
CumDNFRate             0
dtype: int64

Value ranges:
Position: 1.0-20.0
Qualifying: 1.0-20.0
Recent3Avg: 1.0-20.0

Rows: 439 -> 438


In [16]:
# Block 11 修正版: Build model
with pm.Model() as f1_model:
    
    # Constructor effects
    constructor_sd = pm.HalfNormal('constructor_sd', sigma=1.6)
    constructor = pm.Normal('constructor', mu=0, sigma=constructor_sd, 
                           shape=n_constructors)
    
    constructor_season_sd = pm.HalfNormal('constructor_season_sd', sigma=0.7)
    constructor_season = pm.Normal('constructor_season', mu=0, 
                                   sigma=constructor_season_sd,
                                   shape=n_constructor_seasons)
    
    # Driver effects
    driver_sd = pm.HalfNormal('driver_sd', sigma=0.5)
    driver = pm.Normal('driver', mu=0, sigma=driver_sd, shape=n_drivers)
    
    # Feature coefficients
    alpha = pm.Normal('alpha', mu=10.5, sigma=1.0)
    beta_quali = pm.Normal('beta_quali', mu=0.6, sigma=0.2)
    gamma_track = pm.Normal('gamma_track', mu=0, sigma=1.5, shape=3)
    delta_form = pm.Normal('delta_form', mu=0, sigma=2.0)
    epsilon_dnf = pm.Normal('epsilon_dnf', mu=2.0, sigma=1.5)
    
    # Linear predictor - 确保类型正确
    mu = pm.Deterministic('mu',
        alpha +
        constructor[race_data['ConstructorIdx'].values.astype(int)] +
        constructor_season[race_data['ConstructorSeasonIdx'].values.astype(int)] +
        driver[race_data['DriverIdx'].values.astype(int)] +
        beta_quali * race_data['QualifyingPosition'].values.astype(float) +
        gamma_track[race_data['TrackTypeIdx'].values.astype(int)] +
        delta_form * race_data['Recent3Avg'].values.astype(float) +
        epsilon_dnf * race_data['CumDNFRate'].values.astype(float)
    )
    
    # Likelihood
    sigma = pm.HalfNormal('sigma', 1.0)
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma,
                     observed=race_data['EffectivePosition'].values.astype(float))

print("✅ Model built")

# Test model
with f1_model:
    try:
        f1_model.point_logps()
        print("✅ Model logp computable")
    except Exception as e:
        print(f"❌ Model error: {e}")

✅ Model built
✅ Model logp computable


In [17]:
# Block 12 修正版: Prior check (简化)
with f1_model:
    prior_pred = pm.sample_prior_predictive(samples=100, random_seed=42)

# 检查 prior 合理性
prior_mean = prior_pred.prior['y_obs'].mean().values
prior_std = prior_pred.prior['y_obs'].std().values

print(f"Prior mean: {prior_mean:.2f} (should be ~10.5)")
print(f"Prior std: {prior_std:.2f}")

if 5 < prior_mean < 15:
    print("✅ Prior looks reasonable")
else:
    print("⚠️ Prior may need adjustment")

Sampling: [alpha, beta_quali, constructor, constructor_sd, constructor_season, constructor_season_sd, delta_form, driver, driver_sd, epsilon_dnf, gamma_track, sigma, y_obs]


KeyError: "No variable named 'y_obs'. Variables on the dataset include ['chain', 'draw', 'mu_dim_0', 'mu', 'sigma', ..., 'epsilon_dnf', 'gamma_track_dim_0', 'gamma_track', 'driver_dim_0', 'driver']"

In [20]:
# Block 13 修正版: Sample with error handling
with f1_model:
    try:
        trace = pm.sample(
            2000,  # 先减少到 500
            tune=1000,
            target_accept=0.95,  # 降低到 0.85
            chains=4,
            cores=1,  # 先用单核
            return_inferencedata=True,
            random_seed=42
        )
        print("✅ Sampling complete")
    except Exception as e:
        print(f"❌ Sampling error: {e}")
        print("\nTrying to debug...")
        f1_model.debug()

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (4 chains in 1 job)
NUTS: [constructor_sd, constructor, constructor_season_sd, constructor_season, driver_sd, driver, alpha, beta_quali, gamma_track, delta_form, epsilon_dnf, sigma]


Output()

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 41 seconds.
There were 42 divergences after tuning. Increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


✅ Sampling complete


In [21]:
# Block 14: Diagnostics
print("=== CONVERGENCE DIAGNOSTICS ===")
summary = az.summary(trace, var_names=['constructor_sd', 'driver_sd', 
                                        'alpha', 'beta_quali', 'sigma'])
print(summary)

# Check R-hat and ESS
rhat_ok = (summary['r_hat'] < 1.01).all()
ess_ok = (summary['ess_bulk'] > 400).all()

print(f"\n✅ R-hat < 1.01: {rhat_ok}")
print(f"✅ ESS > 400: {ess_ok}")

=== CONVERGENCE DIAGNOSTICS ===
                 mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
constructor_sd  1.687  0.707   0.054    2.811      0.023    0.015     868.0   
driver_sd       0.890  0.307   0.287    1.448      0.009    0.005    1076.0   
alpha           8.834  0.877   7.177   10.433      0.017    0.013    2929.0   
beta_quali      0.509  0.045   0.426    0.595      0.001    0.000    6353.0   
sigma           4.090  0.139   3.831    4.350      0.002    0.002    7805.0   

                ess_tail  r_hat  
constructor_sd     512.0    1.0  
driver_sd          831.0    1.0  
alpha             2548.0    1.0  
beta_quali        5628.0    1.0  
sigma             4743.0    1.0  

✅ R-hat < 1.01: True
✅ ESS > 400: True


In [22]:
# Block 15: Calculate R²
with f1_model:
    ppc = pm.sample_posterior_predictive(trace)

y_pred = ppc.posterior_predictive['y_obs'].mean(dim=['chain', 'draw']).values
y_true = race_data['EffectivePosition'].values

from sklearn.metrics import r2_score
r2 = r2_score(y_true, y_pred)

print(f"\n🎯 Model R² = {r2:.3f}")
print(f"   Target: R² > 0.55")

if r2 > 0.55:
    print("   ✅ GOAL ACHIEVED!")
else:
    print("   ⚠️ Need improvement")

Sampling: [y_obs]


Output()


🎯 Model R² = 0.498
   Target: R² > 0.55
   ⚠️ Need improvement


In [24]:
# Block 18 完整版: Ordinal model
with pm.Model() as ordinal_model:
    
    # Constructor effects
    constructor_sd = pm.HalfNormal('constructor_sd', sigma=1.6)
    constructor = pm.Normal('constructor', mu=0, sigma=constructor_sd, 
                           shape=n_constructors)
    
    constructor_season_sd = pm.HalfNormal('constructor_season_sd', sigma=0.7)
    constructor_season = pm.Normal('constructor_season', mu=0, 
                                   sigma=constructor_season_sd,
                                   shape=n_constructor_seasons)
    
    # Driver effects
    driver_sd = pm.HalfNormal('driver_sd', sigma=0.5)
    driver = pm.Normal('driver', mu=0, sigma=driver_sd, shape=n_drivers)
    
    # Feature coefficients
    alpha = pm.Normal('alpha', mu=10.5, sigma=1.0)
    beta_quali = pm.Normal('beta_quali', mu=0.6, sigma=0.2)
    gamma_track = pm.Normal('gamma_track', mu=0, sigma=1.5, shape=3)
    delta_form = pm.Normal('delta_form', mu=0, sigma=2.0)
    epsilon_dnf = pm.Normal('epsilon_dnf', mu=2.0, sigma=1.5)
    
    # Linear predictor (NEGATIVE - lower = better)
    eta = -(
        alpha +
        constructor[race_data['ConstructorIdx'].values.astype(int)] +
        constructor_season[race_data['ConstructorSeasonIdx'].values.astype(int)] +
        driver[race_data['DriverIdx'].values.astype(int)] +
        beta_quali * race_data['QualifyingPosition'].values +
        gamma_track[race_data['TrackTypeIdx'].values.astype(int)] +
        delta_form * race_data['Recent3Avg'].values +
        epsilon_dnf * race_data['CumDNFRate'].values
    )
    
    # Cutpoints
    cutpoints = pm.Normal('cutpoints', 
                         mu=np.linspace(-3, 3, 19), 
                         sigma=2, 
                         shape=19,
                         transform=pm.distributions.transforms.ordered)
    
    # Ordinal likelihood (position 1-20 → index 0-19)
    y_obs = pm.OrderedLogistic('y_obs', 
                               cutpoints=cutpoints, 
                               eta=eta,
                               observed=race_data['EffectivePosition'].values.astype(int) - 1)

print("✅ Ordinal model built")

# Sample
with ordinal_model:
    trace_ordinal = pm.sample(
        1000,
        tune=1000,
        target_accept=0.95,
        chains=4,
        return_inferencedata=True
    )

✅ Ordinal model built


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [constructor_sd, constructor, constructor_season_sd, constructor_season, driver_sd, driver, alpha, beta_quali, gamma_track, delta_form, epsilon_dnf, cutpoints]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 49 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


In [26]:
# Block 15: Calculate R²
with ordinal_model:
    ppc = pm.sample_posterior_predictive(trace)

y_pred = ppc.posterior_predictive['y_obs'].mean(dim=['chain', 'draw']).values
y_true = race_data['EffectivePosition'].values

from sklearn.metrics import r2_score
r2 = r2_score(y_true, y_pred)

print(f"\n🎯 Model R² = {r2:.3f}")
print(f"   Target: R² > 0.55")

if r2 > 0.55:
    print("   ✅ GOAL ACHIEVED!")
else:
    print("   ⚠️ Need improvement")

Sampling: [cutpoints, y_obs]


Output()


🎯 Model R² = -3.244
   Target: R² > 0.55
   ⚠️ Need improvement
